In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingRegressor, VotingRegressor, StackingRegressor, RandomForestRegressor, BaggingClassifier, VotingClassifier, StackingClassifier, RandomForestClassifier
import requests
import time
try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*a, **k):
        return None
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import mean_squared_error, accuracy_score
import numpy as np
import joblib


In [2]:
df = pd.read_csv('epi_r.csv', low_memory=False)

In [ ]:
load_dotenv()
API_KEY = os.getenv('API_KEY')
CACHE_FILE = os.getenv('CACHE_FILE')

def load_cache():
    if CACHE_FILE and os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_cache(cache):
    if CACHE_FILE:
        with open(CACHE_FILE, 'w') as f:
            json.dump(cache, f)

cache = load_cache()

In [4]:
def is_ingredient(name):
    if name in cache:
        return name, cache[name]
    url = f"https://api.nal.usda.gov/fdc/v1/foods/search?query={name}&api_key={API_KEY}&pageSize=10&dataType=Foundation,SR%20Legacy"
    try:
        response = requests.get(url)
        foods = response.json().get('foods', [])
        for food in foods:
            main_name = food.get('description', '').lower().split(',')[0].strip()
            name_lower = name.lower()
            if main_name == name_lower or name_lower in main_name or main_name in name_lower:
                cache[name] = True
                return name, True
    except:
        pass
    cache[name] = False
    return name, False

In [ ]:
non_ingredients = [
    'alaska', 'alcoholic', 'breakfast', 'buffalo', 'burrito', 'cake',
    'canada', 'cocktail', 'cookie', 'dessert', 'dip', 'fat free',
    'flat bread', 'fry', 'game', 'grill', 'ice cream', 'lasagna',
    'lunch', 'macaroni and cheese', 'meatball', 'meatloaf', 'organic',
    'pancake', 'pasta maker', 'pastry', 'picnic', 'pie', 'pizza',
    'pot pie', 'potato salad', 'poultry sausage', 'roast', 'salad',
    'salad dressing', 'sandwich', 'sauce', 'smoothie', 'steak',
    'stew', 'taco', 'tart', 'waffle', 'vegetarian', 'fruit juice',
    'sugar conscious', 'snack', 'meat', 'fruit', 'spice', 'seed', 'poultry',
]

non_food = set(non_ingredients) | {
    'title', 'rating', 'calories', 'protein', 'fat', 'sodium',
    'leftovers', 'california', 'dominican republic',
    'low cholesterol', 'pan-fry', 'braise',
}

ingredient_columns = [c for c in df.columns if c not in non_food]

df_filtered = df[['rating'] + ingredient_columns]
df_filtered = df_filtered.apply(pd.to_numeric, errors='coerce').fillna(0)
df_filtered.to_csv('epi_r_filtered.csv', index=False)
print('df_filtered shape:', df_filtered.shape)

similar_recipes = df[['title', 'rating']].copy()
similar_recipes['url'] = (
    'https://www.epicurious.com/recipes/food/views/' +
    similar_recipes['title'].str.lower()
    .str.replace(r'[^a-z0-9]+', '-', regex=True)
    .str.strip('-')
)
similar_recipes.to_csv('similar_recipes.csv', index=False)
print('similar_recipes shape:', similar_recipes.shape)

In [6]:
X = df_filtered.drop(columns=['rating']).fillna(0)
Y = df_filtered['rating'].dropna()
X = X.loc[Y.index]

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (14572, 204), Test: (3644, 204)


In [ ]:
models = {
    'LinearRegression': (LinearRegression(), {}),
    'DecisionTree': (DecisionTreeRegressor(random_state=21), {'max_depth': [5, 10, 20, None], 'min_samples_split': [2, 5, 10]}),
    'RandomForest': (RandomForestRegressor(random_state=21, n_jobs=-1), {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, None]}),
    'DummyRegressor': (DummyRegressor(strategy='mean'), {}),
    'Bagging': (BaggingRegressor(estimator=DecisionTreeRegressor(random_state=21), random_state=21, n_jobs=-1), {'n_estimators': [10, 50, 100]}),
    'Voting': (VotingRegressor(estimators=[('dt', DecisionTreeRegressor(random_state=21)), ('rf', RandomForestRegressor(random_state=21))]), {}),
    'Stacking': (StackingRegressor(estimators=[('dt', DecisionTreeRegressor(random_state=21)), ('rf', RandomForestRegressor(random_state=21))], final_estimator=LinearRegression()), {}),
}

results = {}
best_model = {}

In [ ]:
for name, (model, params) in models.items():
    if params:
        grid_search = GridSearchCV(model, params, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
        grid_search.fit(X_train, Y_train)
        model = grid_search.best_estimator_
        best_model[name] = model
        cv_rmse = -grid_search.best_score_
        Y_pred = model.predict(X_test)
    else:
        model.fit(X_train,Y_train)
        Y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
        cv_rmse = -cv_scores.mean()
    rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
    results[name] = {'test RMSE': rmse,'cv RMSE': cv_rmse}

for name, metrics in results.items():
    print(f"{name}: Test RMSE = {metrics['test RMSE']:.4f}, CV RMSE = {metrics['cv RMSE']:.4f}")

best = min(results, key=lambda name: results[name]['cv RMSE'])
print(f"Best model: {best}, CV RMSE = {results[best]['cv RMSE']:.4f}, Test RMSE = {results[best]['test RMSE']:.4f}")

In [8]:
class_models = {
    'LogisticRegression': (LogisticRegression(max_iter=1000), {}),
    'DecisionTree': (DecisionTreeClassifier(random_state=21), {'max_depth': [5, 10, 20, None], 'min_samples_split': [2, 5, 10]}),
    'RandomForest': (RandomForestClassifier(random_state=21, n_jobs=-1), {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, None]}),
    'Dummy': (DummyClassifier(strategy='most_frequent'), {}),
    'Bagging': (BaggingClassifier(estimator=DecisionTreeClassifier(random_state=21), random_state=21, n_jobs=-1), {'n_estimators': [10, 50, 100]}),
    'Voting': (VotingClassifier(estimators=[('dt', DecisionTreeClassifier(random_state=21)), ('rf', RandomForestClassifier(random_state=21))]), {}),
    'Stacking': (StackingClassifier(estimators=[('dt', DecisionTreeClassifier(random_state=21)), ('rf', RandomForestClassifier(random_state=21))], final_estimator=LogisticRegression()), {}),
}

Y_class = Y.round().astype(int)
X_train, X_test, Y_train_c, Y_test_c = train_test_split(X, Y_class, test_size=0.2, random_state=42)

In [9]:
def classification_pipeline(class_models, X_train, X_test, Y_train, Y_test, scoring='accuracy'):
    class_results = {}
    best_class_model = {}
    for name, (model, params) in class_models.items():
        if params:
            grid_search = GridSearchCV(model, params, cv=5, scoring=scoring, n_jobs=-1)
            grid_search.fit(X_train, Y_train)
            best = grid_search.best_estimator_
            best_class_model[name] = best
            Y_pred = best.predict(X_test)
            accuracy = accuracy_score(Y_test, Y_pred)
            class_results[name] = {'test accuracy': accuracy, 'cv accuracy': grid_search.best_score_}
        else:
            model.fit(X_train, Y_train)
            Y_pred = model.predict(X_test)
            accuracy = accuracy_score(Y_test, Y_pred)
            cv_accuracy = cross_val_score(model, X_train, Y_train, cv=5, scoring=scoring, n_jobs=-1).mean()
            class_results[name] = {'test accuracy': accuracy, 'cv accuracy': cv_accuracy}

    for name, metrcis in class_results.items():
        print(f"{name}: Test Accuracy = {metrcis['test accuracy']:.4f}, CV Accuracy = {metrcis['cv accuracy']:.4f}")

    best = max(class_results, key=lambda n: class_results[n]['cv accuracy'])
    print(f"Best: {best}")
    return best_class_model, best

In [10]:
print("Integer classification")
models_int, best_int = classification_pipeline(class_models, X_train, X_test, Y_train_c, Y_test_c, scoring='accuracy')

Integer classification


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:

LogisticRegression: Test Accuracy = 0.7289, CV Accuracy = 0.7228
DecisionTree: Test Accuracy = 0.7267, CV Accuracy = 0.7212
RandomForest: Test Accuracy = 0.7431, CV Accuracy = 0.7277
Dummy: Test Accuracy = 0.7286, CV Accuracy = 0.7228
Bagging: Test Accuracy = 0.7223, CV Accuracy = 0.7156
Voting: Test Accuracy = 0.6863, CV Accuracy = 0.6620
Stacking: Test Accuracy = 0.7489, CV Accuracy = 0.7366
Best: Stacking


In [11]:
def to_category(r):
    if r in [0,1]:
        return 'bad'
    elif r in [2,3]:
        return 'so-so'
    else:
        return 'great'

Y_cat = Y_class.map(to_category)
X_tr_cat, X_te_cat, Y_tr_cat, Y_te_cat = train_test_split(X, Y_cat, test_size=0.2, random_state=42)

In [12]:
print("Categories + accuracy")
models_cat, best_cat = classification_pipeline(class_models, X_tr_cat, X_te_cat, Y_tr_cat, Y_te_cat, 'accuracy')

print("Categories + f1_macro")
models_f1, best_f1 = classification_pipeline(class_models, X_tr_cat, X_te_cat, Y_tr_cat, Y_te_cat, 'f1_macro')

Categories + accuracy


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:

LogisticRegression: Test Accuracy = 0.8749, CV Accuracy = 0.8728
DecisionTree: Test Accuracy = 0.8738, CV Accuracy = 0.8711
RandomForest: Test Accuracy = 0.8847, CV Accuracy = 0.8751
Dummy: Test Accuracy = 0.8749, CV Accuracy = 0.8728
Bagging: Test Accuracy = 0.8721, CV Accuracy = 0.8655
Voting: Test Accuracy = 0.8828, CV Accuracy = 0.8715
Stacking: Test Accuracy = 0.8790, CV Accuracy = 0.8734
Best: RandomForest
Categories + f1_macro


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:

LogisticRegression: Test Accuracy = 0.8749, CV Accuracy = 0.3107
DecisionTree: Test Accuracy = 0.8139, CV Accuracy = 0.4083
RandomForest: Test Accuracy = 0.8845, CV Accuracy = 0.4284
Dummy: Test Accuracy = 0.8749, CV Accuracy = 0.3107
Bagging: Test Accuracy = 0.8721, CV Accuracy = 0.4281
Voting: Test Accuracy = 0.8828, CV Accuracy = 0.4115
Stacking: Test Accuracy = 0.8790, CV Accuracy = 0.3183
Best: RandomForest


In [13]:
joblib.dump(models_f1[best_f1], 'bestmodel.pkl')

['bestmodel.pkl']

Конечный выбор - классификация (RandomForest)

In [14]:
DAILY_VALUES = {
    # DRVs (первая таблица)
    'Total lipid (fat)': 78,           # g
    'Fatty acids, total saturated': 20, # g
    'Cholesterol': 300,                 # mg
    'Carbohydrate, by difference': 275, # g
    'Sodium, Na': 2300,                 # mg
    'Fiber, total dietary': 28,         # g
    'Protein': 50,                      # g
    'Total Sugars': 50, # g (added sugars)
    
    # RDIs (вторая таблица)
    'Vitamin A, RAE': 900,              # mcg
    'Vitamin C, total ascorbic acid': 90,  # mg
    'Calcium, Ca': 1300,                # mg
    'Iron, Fe': 18,                     # mg
    'Vitamin D (D2 + D3)': 20,         # mcg
    'Vitamin E (alpha-tocopherol)': 15, # mg
    'Vitamin K (phylloquinone)': 120,   # mcg
    'Thiamin': 1.2,                     # mg
    'Riboflavin': 1.3,                  # mg
    'Niacin': 16,                       # mg
    'Vitamin B-6': 1.7,                 # mg
    'Folate, total': 400,               # mcg
    'Vitamin B-12': 2.4,               # mcg
    'Biotin': 30,                       # mcg
    'Pantothenic acid': 5,              # mg
    'Phosphorus, P': 1250,              # mg
    'Iodine, I': 150,                   # mcg
    'Magnesium, Mg': 420,              # mg
    'Zinc, Zn': 11,                     # mg
    'Selenium, Se': 55,                 # mcg
    'Copper, Cu': 0.9,                  # mg
    'Manganese, Mn': 2.3,              # mg
    'Chromium, Cr': 35,                # mcg
    'Molybdenum, Mo': 45,              # mcg
    'Chloride, Cl': 2300,              # mg
    'Potassium, K': 4700,              # mg
    'Choline, total': 550,             # mg
}

In [15]:
def get_nutrient_percentage(ingredient):
    try:
        url = f"https://api.nal.usda.gov/fdc/v1/foods/search?query={ingredient}&api_key={API_KEY}&pageSize=50&dataType=Foundation,SR%20Legacy"
        response = requests.get(url)
        foods = response.json().get('foods', [])
        if not foods:
            return None

        ingredient_lower = ingredient.lower()
        target_food = None
        for food in foods:
            main_name = food.get('description', '').lower().split(',')[0].strip()
            desc = food.get('description', '').lower()
            if main_name == ingredient_lower and ('fluid' in desc or 'whole' in desc):
                target_food = food
                break

        if not target_food:
            for food in foods:
                main_name = food.get('description', '').lower().split(',')[0].strip()
                if main_name == ingredient_lower:
                    target_food = food
                    break

        result = {}
        for n in target_food.get('foodNutrients', []):
            name = n.get('nutrientName')
            value = n.get('value', 0)
            if name in DAILY_VALUES:
                result[name] = round(value / DAILY_VALUES[name] * 100, 2)

        return result if result else None
    except:
        return None

In [17]:
print(get_nutrient_percentage('milk'))

{'Vitamin A, RAE': 4.89, 'Total lipid (fat)': 8.97, 'Carbohydrate, by difference': 1.95, 'Calcium, Ca': 14.85, 'Potassium, K': 2.91, 'Zinc, Zn': 4.91, 'Niacin': 2.61, 'Pantothenic acid': 8.14, 'Vitamin B-6': 3.53, 'Protein': 11.96, 'Fiber, total dietary': 0.0, 'Iron, Fe': 0.56, 'Magnesium, Mg': 4.29, 'Phosphorus, P': 12.64, 'Sodium, Na': 1.91, 'Copper, Cu': 5.11, 'Manganese, Mn': 0.78, 'Vitamin C, total ascorbic acid': 4.67, 'Thiamin': 5.42, 'Riboflavin': 27.31, 'Folate, total': 1.75, 'Vitamin B-12': 29.58, 'Cholesterol': 9.0, 'Fatty acids, total saturated': 23.0, 'Selenium, Se': 3.09}


In [ ]:
nutrition_data = {}
ingredients = [c for c in df_filtered.columns if c != 'rating']
for i, ing in enumerate(ingredients):
    result = get_nutrient_percentage(ing)
    if result:
        nutrition_data[ing] = result
    if i % 20 == 0:
        print(f"Processed {i}/{len(ingredients)}")

nutrition_df = pd.DataFrame(nutrition_data).T.fillna(0)
if not nutrition_df.empty:
    nutrition_df.to_csv('nutrition_facts.csv')
    print(nutrition_df.shape)
else:
    print('No nutrition data fetched; keeping existing nutrition_facts.csv')